<a href="https://colab.research.google.com/github/Allah-Bakhsh/flyrank-ml-internship-week1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Allah-Bakhsh/flyrank-ml-internship-week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item, for one client, on one calendar day (grain: `report_date` × `client_hash_id` × `content_hash_id`). Time window for this contract: `month=2026-03`  a mid-panel month, not the sealed final month (`2026-06`), to avoid building label logic on the natural outcome window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `report_date`/count of days observed all directly measured, available at the decision moment.

**Label/proxy:** built from `gsc_avg_position` and CTR (clicks/impressions) no direct "needs refresh" label exists, so this is a defined proxy, not an observed outcome.

**Context (not modeled directly):** `client_hash_id`, `client_has_gsc`, `client_has_ga4` used to filter/group, not as predictive features.

**Excluded:** all `ga4_*` and `sessions_*`/`ai_*` fields. Why: `client_has_ga4` is `False` for a large share of rows in this sample, so GA4-based features would only exist for a subset of clients using them would bias the model toward GA4-connected clients and silently exclude everyone else.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three queries below prove the grain, the slice's size/date span, and GSC availability (filtered with `IS TRUE`). Then a five-feature frame is built from the same month, and the leakage trap is run and reversed.

In [3]:
# --- Setup (connect to Hugging Face) ---
from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")

import duckdb
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}');")

MARCH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

# --- Query 1: grain check ---
grain_check = con.sql(f"""
    WITH combos AS (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
        FROM '{MARCH}'
        GROUP BY 1, 2, 3
    )
    SELECT COUNT(*) AS distinct_combos, MAX(n) AS max_rows_per_combo
    FROM combos
""").df()
print("Query 1 — Grain check (max_rows_per_combo should be 1):")
display(grain_check)

# --- Query 2: row count + date span for this slice ---
slice_stats = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT content_hash_id) AS distinct_pages
    FROM '{MARCH}'
    WHERE client_has_gsc IS TRUE
""").df()
print("\nQuery 2 — Row count and date span, GSC-enabled clients only:")
display(slice_stats)

# --- Query 3: availability check with IS TRUE ---
availability = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows
    FROM '{MARCH}'
""").df()
print("\nQuery 3 — Availability (rows where gsc_data_available IS TRUE):")
display(availability)

# --- Five features, built from the same month ---
features = con.sql(f"""
    SELECT
        content_hash_id,
        COUNT(*) AS days_observed,
        AVG(gsc_impressions) AS avg_gsc_impressions,
        AVG(gsc_clicks) AS avg_gsc_clicks,
        AVG(gsc_avg_position) AS avg_gsc_position,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr
    FROM '{MARCH}'
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

print("\nFive-feature frame (one row = one content page for this month):")
display(features.head())
print("""
Feature availability notes:
- days_observed: knowable at the decision moment because it only counts days already logged.
- avg_gsc_impressions: knowable because it's a historical average over days already observed.
- avg_gsc_clicks: same — historical average, no future data used.
- avg_gsc_position: same — historical average of daily reported position.
- ctr: knowable because it's computed purely from this month's already-observed clicks/impressions.
""")

# --- THE TRAP: add ONE label-derived column on purpose ---
# Simpler, cleaner label: just position, avoids the zero-inflated CTR problem
features["label_refresh_candidate"] = (
    features["avg_gsc_position"] > features["avg_gsc_position"].median()
).astype(int)

print("Label balance check:")
print(features["label_refresh_candidate"].value_counts(normalize=True))

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

honest_cols = ["days_observed", "avg_gsc_impressions", "avg_gsc_clicks"]
X = features[honest_cols].fillna(0)
y = features["label_refresh_candidate"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
honest_score = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train).score(X_test, y_test)
print(f"\nHonest accuracy, features NOT derived from position: {honest_score:.3f}")
print(f"(Majority-class baseline, for comparison: {max(y.mean(), 1-y.mean()):.3f})")

# Smuggle avg_gsc_position back in — this is literally what the label was built from
features["leaky_position_flag"] = features["avg_gsc_position"]

X_leak = features[honest_cols + ["leaky_position_flag"]].fillna(0)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leak, y, test_size=0.3, random_state=42)
leaky_score = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train_l, y_train_l).score(X_test_l, y_test_l)
print(f"Leaky accuracy, with avg_gsc_position smuggled back in: {leaky_score:.3f}  <- jumps toward perfect, this is the trap")

features = features.drop(columns=["leaky_position_flag"])
print(f"\nLeaky column removed. Keeping the honest number: {honest_score:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 — Grain check (max_rows_per_combo should be 1):


,distinct_combos,max_rows_per_combo
0,9841378,1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Query 2 — Row count and date span, GSC-enabled clients only:


...

,total_rows,gsc_available_rows
0,9841378,3611061.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Five-feature frame (one row = one content page for this month):


,content_hash_id,days_observed,avg_gsc_impressions,avg_gsc_clicks,avg_gsc_position,ctr
0,content_2e6360ad20fd7107,31,29.000000,0.032258,5.145765,0.001112
1,content_ac8663da7484669a,17,2.000000,0.000000,4.909314,0.000000
2,content_d49a012dcb924e31,31,10.612903,0.000000,5.177774,0.000000
3,content_614baf2af4330bd7,31,24.903226,0.032258,4.685335,0.001295
4,content_4a1ca0fa5c177e0c,10,1.400000,0.000000,4.266667,0.000000



Feature availability notes:
- days_observed: knowable at the decision moment because it only counts days already logged.
- avg_gsc_impressions: knowable because it's a historical average over days already observed.
- avg_gsc_clicks: same — historical average, no future data used.
- avg_gsc_position: same — historical average of daily reported position.
- ctr: knowable because it's computed purely from this month's already-observed clicks/impressions.

Label balance check:
label_refresh_candidate
0    0.5
1    0.5
Name: proportion, dtype: float64

Honest accuracy, features NOT derived from position: 0.638
(Majority-class baseline, for comparison: 0.500)
Leaky accuracy, with avg_gsc_position smuggled back in: 1.000  <- jumps toward perfect, this is the trap

Leaky column removed. Keeping the honest number: 0.638


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice is GSC-only by design (GA4 excluded due to sparse client coverage), so it can't say anything about actual user behavior on the page (engagement, scroll depth, conversions) only search visibility. It also can't establish causality: a page appearing as a "refresh candidate" this month is a pattern match on position/CTR, not proof that refreshing it would improve anything. Client coverage is uneven (`client_has_gsc` isn't True for every row), so this slice systematically underrepresents clients without GSC connected.

Additional limitation observed directly while building the leakage check: a label built purely from CTR collapsed to almost all-zero, because most pages have zero clicks in a given month — CTR-based labels need careful handling of this zero-inflation. The position-based label used instead was properly balanced (50/50) and gave an honest baseline of 0.638 vs. a leaky 1.000 when position was smuggled back in as a feature — a clear, real demonstration of the leakage trap on this slice.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.